# Учебное исследование территории Ивангорода с помощью BlocksNet

Этот ноутбук шаг за шагом исследует город **Ивангород** (Ленинградская область) при помощи открытой библиотеки
**BlocksNet** (версия 1.0.0a9) и открытых данных **OpenStreetMap**.

Простым языком, мы:

1. скачаем границу города и уличную сеть из OpenStreetMap;
2. «нарежем» город на **кварталы** — небольшие участки, окружённые дорогами;
3. скачаем готовый **пешеходный граф** и посчитаем, сколько **минут пешком** идти от каждого квартала до каждого;
4. соберём данные о зданиях (и оценим население кварталов) и о школах;
5. посчитаем показатели: обеспеченность школами, плотность застройки, морфотипы, доступность территории;
6. сохраним всё в папку `examples/data/ivangorod` — при повторном запуске данные не скачиваются, а берутся из файлов.

**Разделы ноутбука:**

1. Настройки
2. Границы города
3. Дороги, железные дороги и вода
4. Нарезка кварталов
5. Типы использования кварталов (ленд-юз)
6. **🛑 Остановка перед расчётами** (важный момент — не пропустите!)
7. Пешеходный граф
8. Матрица времени в пути
9. Здания и население
10. Сервисы: школы
11. Показатели
12. Итоговый слой и проверка результатов
13. Повторный запуск и «если что-то не работает»


## Как пользоваться ноутбуком

- Запускайте ячейки **по порядку сверху вниз**: выделите ячейку и нажмите **Shift+Enter** (или треугольник «выполнить» слева от ячейки).
- **Первый запуск — самый долгий** (несколько минут): данные скачиваются из OpenStreetMap. Все последующие запуски быстрые, потому что готовые данные читаются из файлов.
- Менять код не нужно. Все файлы сохраняются автоматически в папку `data/ivangorod` рядом с ноутбуком. Если запустить ноутбук заново, он найдёт эти файлы и не станет скачивать данные повторно.
- Если где-то появится ошибка (красный текст) — не пугайтесь: в конце ноутбука есть раздел «Если что-то не работает».


## Раздел 1. Настройки

Первая ячейка загружает нужные библиотеки и создаёт папку для данных и результатов.
Всё, что она печатает (версии программ и путь к папке), полезно для проверки.


In [ ]:
%matplotlib inline

from pathlib import Path
import pickle
import importlib.metadata as md

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd

print('BlocksNet:', md.version('blocksnet'), '| IduEdu:', md.version('iduedu'), '| OSMnx:', md.version('osmnx'))

# Папка, куда сохраняются все скачанные данные и результаты расчётов
DATA_DIR = Path('data/ivangorod')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Папка для данных и результатов:', DATA_DIR.resolve())


Вторая ячейка — две вспомогательные функции. Они нужны, чтобы работает главный приём ноутбука:
**«если файл уже есть — читаем его, если нет — скачиваем/считаем»**. Это и есть повторное использование данных.


In [ ]:
def save_layer(layer, name, with_block_id=False):
    """Сохраняет слой (таблицу с геометрией) в файл GeoJSON в папке data/ivangorod.

    GeoJSON — стандартный формат для карт: его можно открыть в QGIS, на веб-карте и в других программах.
    Результат всегда сохраняется в системе координат EPSG:4326 (стандарт для GeoJSON).
    Параметр with_block_id=True добавляет колонку с номерами кварталов, чтобы при повторном
    чтении файла таблицы правильно совмещались друг с другом.
    """
    layer = layer.copy()
    if with_block_id:
        layer['block_id'] = layer.index.astype(int)
    layer = layer.reset_index(drop=True)
    path = DATA_DIR / f'{name}.geojson'
    layer.to_crs(4326).to_file(path, driver='GeoJSON', index=False)
    print('Сохранено:', path)


def load_layer(name):
    """Читает ранее сохранённый слой, если файл существует. Если файла нет — возвращает None,
    и ячейка, которая вызвала функцию, сама скачает данные из OpenStreetMap или посчитает результат заново."""
    path = DATA_DIR / f'{name}.geojson'
    if not path.exists():
        return None
    layer = gpd.read_file(path)
    if 'block_id' in layer.columns:
        layer = layer.set_index('block_id')
        layer.index.name = None
    print('Используем сохранённый файл (скачивание не нужно):', path)
    return layer


## Раздел 2. Границы города

Скачаем границу Ивангорода из OpenStreetMap по названию. Граница — это «рамка», внутри которой дальше
будет считаться всё исследование.

Ячейка ниже: если файл `boundaries.geojson` уже сохранён — он будет прочитан, иначе граница скачается
из OpenStreetMap. Карта покажет, что мы нашли. Площадь Ивангорода — примерно 7–8 км².


In [ ]:
boundaries = load_layer('boundaries')

if boundaries is None:
    boundaries = ox.geocode_to_gdf('Ивангород, Ленинградская область, Россия')
    save_layer(boundaries, 'boundaries')

area_km2 = boundaries.to_crs(boundaries.estimate_utm_crs()).area.sum() / 1_000_000
print('Площадь территории по границе, км²:', round(area_km2, 1))
boundaries.plot().set_axis_off()


## Раздел 3. Дороги, железные дороги и вода

Чтобы поделить город на кварталы, нужны линии (дороги и железные дороги) и участки воды
(река Нарва и другие водоёмы). Скачаем их из OpenStreetMap по границе города.

Набор тегов OSM ниже взят из учебного примера BlocksNet (`pipeline.ipynb`).


In [ ]:
bc_tags = {
    'roads': {
        'highway': ['construction', 'crossing', 'living_street', 'motorway', 'motorway_link', 'motorway_junction',
                    'pedestrian', 'primary', 'primary_link', 'raceway', 'residential', 'road', 'secondary',
                    'secondary_link', 'services', 'tertiary', 'tertiary_link', 'track', 'trunk', 'trunk_link',
                    'turning_circle', 'turning_loop', 'unclassified'],
        'service': ['living_street', 'emergency_access'],
    },
    'railways': {
        'railway': 'rail',
    },
    'water': {
        'riverbank': True,
        'reservoir': True,
        'basin': True,
        'dock': True,
        'canal': True,
        'pond': True,
        'natural': ['water', 'bay'],
        'waterway': ['river', 'canal', 'ditch'],
        'landuse': 'basin',
        'water': 'lake',
    },
}


In [ ]:
polygon_for_osm = boundaries.union_all()  # граница города одним полигоном — так уходит запрос к OpenStreetMap

water = load_layer('water')
if water is None:
    water = ox.features_from_polygon(polygon_for_osm, bc_tags['water'])
    water = water[water.geom_type.isin(['Polygon', 'MultiPolygon', 'LineString', 'MultiLineString'])].copy()
    save_layer(water, 'water')

roads = load_layer('roads')
if roads is None:
    roads = ox.features_from_polygon(polygon_for_osm, bc_tags['roads'])
    roads = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])].copy()
    save_layer(roads, 'roads')

railways = load_layer('railways')
if railways is None:
    railways = ox.features_from_polygon(polygon_for_osm, bc_tags['railways'])
    railways = railways[railways.geom_type.isin(['LineString', 'MultiLineString'])].copy()
    save_layer(railways, 'railways')

print('Объектов: вода —', len(water), '| дороги —', len(roads), '| железные дороги —', len(railways))


## Раздел 4. Нарезка кварталов

**Квартал** — это участок городской территории, окружённый дорогами (а также водой или железной дорогой).
BlocksNet автоматически «режет» город на такие кварталы. Дальше все измерения делаем по кварталам.

Сначала маленькая техническая ячейка: переводим координаты из градусов в метры — так точнее считаются
площади и расстояния. Затем — подготовка объектов и сама нарезка.


In [ ]:
crs = boundaries.estimate_utm_crs()  # подходящая система координат «в метрах» для Ивангорода
print('Система координат для расчётов:', crs)

for gdf in [water, roads, railways, boundaries]:
    gdf.to_crs(crs, inplace=True)


In [ ]:
from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks

lines, polygons = preprocess_urban_objects(roads, railways, water)


In [ ]:
blocks = load_layer('blocks')

if blocks is None:
    blocks = cut_urban_blocks(boundaries, lines, polygons)
    save_layer(blocks, 'blocks', with_block_id=True)
else:
    blocks = blocks.to_crs(crs)

print('Получилось кварталов:', len(blocks))
blocks.plot().set_axis_off()


## Раздел 5. Тип использования кварталов (ленд-юз)

Каждому кварталу назначим «тип использования» на основе зон `landuse` из OpenStreetMap:
жилой, деловой, промышленный, рекреационный и т.д. Правила перевода зон в типы — как в учебном
примере BlocksNet.


In [ ]:
functional_zones = load_layer('functional_zones')

if functional_zones is None:
    zones_osm = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'landuse': True})
    functional_zones = zones_osm.reset_index(drop=True)[['geometry', 'landuse']]
    functional_zones = functional_zones.rename(columns={'landuse': 'functional_zone'})
    save_layer(functional_zones, 'functional_zones')

functional_zones = functional_zones.to_crs(crs)
functional_zones.head()


In [ ]:
from blocksnet.enums import LandUse

rules = {
    'commercial': LandUse.BUSINESS,
    'industrial': LandUse.INDUSTRIAL,
    'cemetery': LandUse.SPECIAL,
    'garages': LandUse.INDUSTRIAL,
    'residential': LandUse.RESIDENTIAL,
    'retail': LandUse.BUSINESS,
    'grass': LandUse.RECREATION,
    'farmland': LandUse.AGRICULTURE,
    'construction': LandUse.SPECIAL,
    'brownfield': LandUse.INDUSTRIAL,
    'forest': LandUse.RECREATION,
    'recreation_ground': LandUse.RECREATION,
    'religious': LandUse.SPECIAL,
    'flowerbed': LandUse.RECREATION,
    'military': LandUse.SPECIAL,
    'landfill': LandUse.TRANSPORT,
}


In [ ]:
from blocksnet.blocks.assignment import assign_land_use

blocks_from_file = load_layer('blocks_landuse')
if blocks_from_file is not None:
    blocks = blocks_from_file.to_crs(crs)
else:
    blocks = assign_land_use(blocks, functional_zones.reset_index(drop=True), rules)
    save_layer(blocks, 'blocks_landuse', with_block_id=True)

ax = blocks.plot(color='#ddd')
blocks.plot(column='land_use', legend=True, ax=ax)
ax.set_axis_off()


---

# 🛑 ОСТАНОВКА ПЕРЕД РАСЧЁТАМИ

**Кварталы подготовлены.** Вы нарезали город на кварталы и назначили им тип использования — это «фундамент» исследования.

Дальше начинаются **расчёты**:

- время в пути между кварталами (пешеходный граф);
- население кварталов (по зданиям) и школы;
- показатели: обеспеченность школами, плотность, морфотипы, доступность территории.

**Прежде чем продолжать:** уточните, какие показатели вам нужны в итоговом отчёте. Ячейки ниже посчитают
всё, что есть в учебном примере `pipeline.ipynb` библиотеки BlocksNet. Если нужен другой или сокращённый
набор — скажите, и мы поправим ноутбук.

**Готовы? Продолжайте запускать ячейки по порядку.**

---


## Раздел 6. Пешеходный граф и время в пути

**Что такое граф.** Это схема дорожек и перекрёстков: точки (перекрёстки) соединены отрезками путей.
У каждого отрезка есть длина (метры) и время прохода (минуты).

**Что скачиваем.** Готовый **пешеходный граф** из **OpenStreetMap**: библиотека BlocksNet получает данные
из OpenStreetMap автоматически (через встроенный модуль IduEdu) и сама считает время каждого отрезка
по его длине и средней скорости пешехода — по умолчанию это **5 км/ч** (значение задано в настройках библиотеки IduEdu). Время хранится в колонке `time_min` — **в минутах**.

**Способ передвижения — пешком.** Интермодальный граф (с общественным транспортом) в этом исследовании
не строим. Если понадобится вариант «на автомобиле» — в конце ноутбука есть инструкция, как поменять один параметр.

Граф скачивается для границы города с небольшим запасом вокруг — так корректнее соединяются пути на окраинах.


In [ ]:
from blocksnet.relations import get_accessibility_graph

graph_file = DATA_DIR / 'graph_walk.pickle'

if graph_file.exists():
    with open(graph_file, 'rb') as f:
        graph = pickle.load(f)
    print('Граф загружен из сохранённого файла:', graph_file)
else:
    graph = get_accessibility_graph(boundaries, 'walk')
    with open(graph_file, 'wb') as f:
        pickle.dump(graph, f)
    print('Пешеходный граф скачан из OpenStreetMap и сохранён:', graph_file)

print('Перекрёстков (узлов):', graph.number_of_nodes())
print('Отрезков путей (рёбер):', graph.number_of_edges())


Сохраним рёбра графа отдельным файлом GeoJSON — их будет удобно показывать на карте
(в том числе на будущем HTML-дашборде). И нарисуем общую картину пешеходных путей.


In [ ]:
from shapely.geometry import LineString

edges_gdf = load_layer('graph_walk_edges')

if edges_gdf is None:
    rows = []
    for u, v, data in graph.edges(data=True):
        rows.append({
            'time_min': data.get('time_min'),
            'length_meter': data.get('length_meter'),
            'geometry': LineString([
                (graph.nodes[u]['x'], graph.nodes[u]['y']),
                (graph.nodes[v]['x'], graph.nodes[v]['y']),
            ]),
        })
    edges_gdf = gpd.GeoDataFrame(rows, crs=graph.graph['crs'])
    save_layer(edges_gdf, 'graph_walk_edges')

edges_gdf.plot(linewidth=0.5, color='black').set_axis_off()


In [ ]:
edge_times = edges_gdf['time_min']
edge_lengths = edges_gdf['length_meter']

print('Медианное время одного отрезка, мин:', round(edge_times.median(), 2))
print('Общая длина пешеходных путей, км:', round(edge_lengths.sum() / 1000, 1))
if edge_times.sum() > 0:
    speed_kmh = edge_lengths.sum() / 1000 / (edge_times.sum() / 60)
    print('Средняя скорость, заложенная в графе, км/ч:', round(speed_kmh, 1))


## Раздел 7. Матрица времени в пути между кварталами

**Матрица доступности** — это таблица размера «кварталы × кварталы», в которой записано, сколько минут
идти от одного квартала до другого (по пешеходным путям). На диагонали — нули (от квартала к нему самому).
Значения считаются от «центральной точки» каждого квартала.

Матрицу сохраняем в два файла: `acc_mx.parquet` (для повторного использования программой) и `acc_mx.csv`
(чтобы можно было открыть таблицу и посмотреть глазами). В GeoJSON матрицу сохранить нельзя:
GeoJSON — формат для карт, а матрица — просто таблица чисел.


In [ ]:
from blocksnet.relations import calculate_accessibility_matrix

mx_file = DATA_DIR / 'acc_mx.parquet'

if mx_file.exists():
    acc_mx = pd.read_parquet(mx_file)
    print('Матрица загружена из сохранённого файла:', mx_file)
else:
    acc_mx = calculate_accessibility_matrix(blocks, graph)
    acc_mx.to_parquet(mx_file)
    acc_mx.to_csv(DATA_DIR / 'acc_mx.csv')
    print('Матрица рассчитана и сохранена:', mx_file)

acc_mx.iloc[:5, :5]


In [ ]:
values = acc_mx.to_numpy().astype(float)
mask = np.isfinite(values) & (values > 0)

print('Пар «квартал → квартал»:', int(mask.sum()))
print('Медианное время в пути, мин:', round(float(np.median(values[mask])), 1))
print('Максимальное время в пути, мин:', round(float(values[mask].max()), 1))


## Раздел 8. Здания и население

Скачаем контуры зданий из OpenStreetMap и определим:

- **жилое** ли здание (колонка `is_living`);
- **число этажей** (колонка `building:levels` в OSM).

Затем библиотека восстанавливает отсутствующие сведения (например, площадь квартир) и оценивает
**население** каждого здания и квартала. Оценка приблизительная — по открытым данным.


In [ ]:
buildings_raw = load_layer('buildings_raw')

if buildings_raw is None:
    buildings_raw = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'building': True})
    buildings_raw = buildings_raw.reset_index(drop=True).to_crs(crs)

    is_living_tags = ['residential', 'house', 'apartments', 'detached', 'terrace', 'dormitory']
    buildings_raw['is_living'] = buildings_raw['building'].apply(lambda b: b in is_living_tags)
    if 'building:levels' in buildings_raw.columns:
        buildings_raw['number_of_floors'] = pd.to_numeric(buildings_raw['building:levels'], errors='coerce')
    else:
        buildings_raw['number_of_floors'] = np.nan

    save_layer(buildings_raw, 'buildings_raw')
else:
    buildings_raw = buildings_raw.to_crs(crs)

print('Зданий скачано:', len(buildings_raw))
buildings_raw.plot().set_axis_off()


In [ ]:
from blocksnet.blocks.aggregation import aggregate_objects
from blocksnet.preprocessing.imputing import impute_buildings

buildings_blocks = load_layer('buildings_blocks')

if buildings_blocks is None:
    buildings = impute_buildings(buildings_raw, default_living_demand=30)
    buildings_blocks = aggregate_objects(blocks, buildings)[0]
    save_layer(buildings_blocks, 'buildings_blocks', with_block_id=True)
else:
    buildings_blocks = buildings_blocks.to_crs(crs)

print('Суммарное население по расчёту, чел.:', int(round(buildings_blocks['population'].sum())))
buildings_blocks.plot('population', legend=True).set_axis_off()


## Раздел 9. Сервисы: школы

Как пример сервиса возьмём **школы** (тег `amenity=school` в OpenStreetMap). Библиотека восстановит
пропущенные сведения (например, вместимость) и посчитает суммарную вместимость школ по кварталам.


In [ ]:
from blocksnet.preprocessing.imputing import impute_services

schools_blocks = load_layer('schools_blocks')

if schools_blocks is None:
    schools = load_layer('schools_raw')
    if schools is None:
        schools = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'amenity': 'school'})
        schools = schools.reset_index(drop=True).to_crs(crs)
        save_layer(schools, 'schools_raw')
    else:
        schools = schools.to_crs(crs)

    schools = impute_services(schools, 'school')
    schools_blocks = aggregate_objects(blocks, schools)[0]
    save_layer(schools_blocks, 'schools_blocks', with_block_id=True)
else:
    schools_blocks = schools_blocks.to_crs(crs)

print('Суммарная вместимость школ, мест:', round(schools_blocks['capacity'].sum()))
schools_blocks.plot('capacity', legend=True).set_axis_off()


## Раздел 10. Показатели

Теперь посчитаем показатели — это и есть «результаты исследования»:

1. **Обеспеченность школами** — сколько мест в школах приходится на жителей, с учётом конкуренции между кварталами (нормативы библиотеки; значение от 0 до 1).
2. **Показатели плотности** — по зданиям внутри кварталов.
3. **Морфотипы застройки** — тип городской среды по двум методикам (Spacematrix и Strelka).
4. **Доступность территории** — как быстро в среднем можно добраться от квартала до остальных.

Это полный набор из учебного примера `pipeline`. Если нужен другой набор метрик — скажите, добавим или уберём.


In [ ]:
from blocksnet.analysis.provision import competitive_provision
from blocksnet.config import service_types_config

df = buildings_blocks[['population']].join(schools_blocks[['capacity']])
_, demand, accessibility = service_types_config['school'].values()
print('Норматив спроса (мест на человека):', demand, '| Нормативная доступность, мин:', accessibility)

prov_df, links_df = competitive_provision(df, acc_mx, accessibility, demand)

prov_df.to_csv(DATA_DIR / 'provision_school.csv')
links_df.to_csv(DATA_DIR / 'provision_school_links.csv')
prov_df.head()


In [ ]:
ax = blocks.plot(color='#ddd')
blocks.join(prov_df).plot('provision_strong', ax=ax, cmap='RdYlGn', vmin=0, vmax=1, legend=True)
ax.set_axis_off()


In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators

buildings_blocks['site_area'] = buildings_blocks.area
density_df = calculate_density_indicators(buildings_blocks)
density_df.head()


In [ ]:
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes, get_strelka_morphotypes

spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(density_df)
spacematrix_df.head()


In [ ]:
ax = blocks.plot(color='#ddd')
blocks.join(spacematrix_df).plot('morphotype', legend=True, ax=ax)
ax.set_axis_off()


In [ ]:
strelka_df = get_strelka_morphotypes(density_df)

ax = blocks.plot(color='#ddd')
blocks.join(strelka_df).plot('morphotype', legend=True, ax=ax)
ax.set_axis_off()


In [ ]:
from blocksnet.analysis.accessibility import area_accessibility

area_acc = area_accessibility(acc_mx, buildings_blocks)
blocks.join(area_acc).plot('area_accessibility', legend=True).set_axis_off()


## Раздел 11. Итог: единый слой показателей

Собираем все показатели в одну таблицу-слой и сохраняем в `blocks_results.geojson`.
Этот файл станет основой для будущей интерактивной HTML-карты с дашбордом.


In [ ]:
results = blocks.copy()
results = results.join(buildings_blocks[['population']])
results = results.join(prov_df)
results = results.join(spacematrix_df[['morphotype']].rename(columns={'morphotype': 'morphotype_spacematrix'}))
results = results.join(strelka_df[['morphotype']].rename(columns={'morphotype': 'morphotype_strelka'}))
results = results.join(area_acc)
results = results.join(density_df.add_prefix('density_'))

save_layer(results, 'blocks_results', with_block_id=True)

summary = pd.Series({
    'Кварталов': len(results),
    'Население, чел.': int(round(results['population'].sum())),
    'Средняя обеспеченность школами (0…1)': round(results['provision_strong'].mean(), 2),
    'Средняя доступность территории, мин': round(results['area_accessibility'].mean(), 1),
})
summary


## Раздел 12. Проверка результатов

Пробегитесь по списку — всё ли похоже на правду:

- ✅ **Границы** (Раздел 2): полигон похож на Ивангород; площадь примерно 7–8 км².
- ✅ **Кварталы** (Раздел 4): визуально совпадают с уличной сетью; всего их десятки–сотни.
- ✅ **Ленд-юз** (Раздел 5): кварталы раскрашены по типам использования, легенда читаема.
- ✅ **Граф** (Раздел 6): рёбер больше, чем узлов; медианное время отрезка — единицы минут.
- ✅ **Матрица** (Раздел 7): медианное время в пути разумно для малого города (обычно меньше 30 минут).
- ✅ **Население** (Раздел 8): суммарно порядка 9–10 тысяч человек (оценка по OSM; официальные данные ≈ 9,3 тыс.).
- ✅ **Школы** (Раздел 9): школы на карте стоят там, где школы в реальности.
- ✅ **Показатели** (Разделы 10–11): обеспеченность в диапазоне 0…1; морфотипы и доступность отображаются на картах.
- ✅ **Файлы**: в `examples/data/ivangorod/` лежат все GeoJSON, `acc_mx.parquet`, `acc_mx.csv`, граф и сводные CSV.

Если какой-то пункт выглядит странно — напишите номер раздела, и мы разберёмся.


## Раздел 13. Повторный запуск и «если что-то не работает»

### Как запустить ноутбук заново

1. Откройте ноутбук (в VS Code или в Jupyter).
2. Выберите меню **Kernel → Restart Kernel and Run All** (в VS Code — **Restart**, затем **Run All**).
3. Повторные запуски быстрые: все данные уже лежат в папке `examples/data/ivangorod` и не скачиваются повторно.

### Как обновить данные

Удалите нужный файл из `examples/data/ivangorod` (например, `roads.geojson`) и запустите ноутбук заново —
этот слой скачается свежий. Чтобы начать «с чистого листа», удалите всю папку `examples/data/ivangorod`.

### Если что-то не работает

- **Ошибка при скачивании (timeout, 429, 504)** — сервер OpenStreetMap (Overpass) иногда перегружен.
  Подождите пару минут и запустите ячейку ещё раз. То, что уже скачано, сохраняется: повторно оно не скачивается.
- **Нашли не тот город** — в Разделе 2 замените строку запроса, например на `'Ivangorod, Leningrad Oblast, Russia'`,
  удалите файл `boundaries.geojson` и запустите раздел заново.
- **Пустой слой** (например, железные дороги) — это нормально, если их нет в данных OSM. Расчёт продолжится.
- **Красная ошибка в середине** — не запускайте ячейки ниже наугад: скопируйте текст ошибки и покажите нам.

### Как посчитать «на автомобиле» вместо «пешком» (по желанию)

1. В Разделе 6 в строке `graph = get_accessibility_graph(boundaries, 'walk')` замените `'walk'` на `'drive'`,
   а имя файла `graph_walk.pickle` — на `graph_drive.pickle` (в двух местах: чтение и запись).
2. Удалите файлы `acc_mx.parquet` и `acc_mx.csv` — матрица пересчитается по дорожной сети.
3. Запустите ноутбук заново.

### Что дальше

Позже мы соберём из этих результатов **одну HTML-страницу**: интерактивную карту города с показателями,
графиками и таблицами (отдельным файлом, на основе `blocks_results.geojson`).
